# Day 04 — Running Your First Evaluation

**Module 1 · Foundations**

We now have all the pieces we need:

- `LLMTestCase` — what we want to evaluate
- LLM Judge — who evaluates it
- Metric — what we evaluate
- Threshold — what score is acceptable

Today we connect them and run our first evaluation.

By the end of this notebook, you will see:

```text
Test Case
    ↓
Metric
    ↓
LLM Judge
    ↓
Score + Reason
    ↓
Threshold
    ↓
PASS / FAIL

In [20]:
import os

from dotenv import load_dotenv
from deepeval import evaluate
from deepeval.models import LocalModel
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

judge = LocalModel(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: openai/gpt-oss-120b (Local Model)


## 2. Create Our Evaluation Metric

Our first metric is **Answer Relevancy**.

It evaluates whether the generated answer is relevant to the user's question.

For today, we will use:

```text
Threshold = 0.5

In [21]:
metric = AnswerRelevancyMetric(
    model=judge,
    threshold=0.5,
)

print("Threshold:", metric.threshold)

Threshold: 0.5


## 3. Create Two Test Cases

We will deliberately create:

1. A relevant answer.
2. An irrelevant answer.

This makes the evaluation behavior easy to observe.

In [22]:
test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="We offer full refunds within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="Our mascot is a llama. Llamas are great. Shipping takes 2 days.",
    ),
]

for i, test_case in enumerate(test_cases, start=1):
    print(f"Test Case {i}")
    print("Input :", test_case.input)
    print("Output:", test_case.actual_output)
    print()

Test Case 1
Input : What's your refund policy?
Output: We offer full refunds within 30 days of purchase.

Test Case 2
Input : What's your refund policy?
Output: Our mascot is a llama. Llamas are great. Shipping takes 2 days.



## 4. Run the Evaluation

Now we connect:

```text
Test Cases + Metric + Judge
            ↓
         evaluate()
            ↓
         Results

In [23]:
results = evaluate(
    test_cases=test_cases,
    metrics=[metric],
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-120b (Local Model), 
strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What's your refund policy?                                                             │
│  │     Actual Output:    Our mascot is a llama. Llamas are great. Shipping takes 2 days.                        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the output only mentioned a     │
│              │                  │       │           │ llama mascot, praised llamas, and discussed shipping      │
│              │                  │       │           │ time—none of which address the refund policy question.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.50                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=423842;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.3s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## 5. Read the Evaluation Results

For each test case, we want to know three things:

- **Score** — how well did it perform?
- **Reason** — why did the judge give that score?
- **Success** — did it pass the threshold?

In [24]:
for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(f"\n{'=' * 50}")
    print(f"Test Case: {result.name}")
    print(f"Score:     {metric_result.score:.2f}")
    print(f"Success:   {metric_result.success}")
    print(f"Reason:    {metric_result.reason}")


Test Case: test_case_0
Score:     1.00
Success:   True
Reason:    The score is 1.00 because the response directly answered the refund policy question with no irrelevant statements, achieving perfect relevance.

Test Case: test_case_1
Score:     0.00
Success:   False
Reason:    The score is 0.00 because the output only mentioned a llama mascot, praised llamas, and discussed shipping time—none of which address the refund policy question.


## 6. What Just Happened?

The evaluation followed this process:

```text
LLMTestCase
     ↓
Answer Relevancy Metric
     ↓
LLM Judge
     ↓
Score + Reason
     ↓
Compare with threshold
     ↓
PASS / FAIL

## 7. Thresholds Control the Verdict

The score and the threshold are different things.

For example:

```text
Score      = 0.72
Threshold  = 0.50
Result     = PASS

In [25]:
metric.threshold = 0.8

results = evaluate(
    test_cases=test_cases,
    metrics=[metric],
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(
        f"{result.name}: "
        f"score={metric_result.score:.2f}, "
        f"success={metric_result.success}"
    )

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-120b (Local Model), 
strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What's your refund policy?                                                             │
│  │     Actual Output:    Our mascot is a llama. Llamas are great. Shipping takes 2 days.                        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.80      │ The score is 0.00 because the response only mentioned a   │
│              │                  │       │           │ llama mascot, praised llamas, and discussed shipping      │
│              │                  │       │           │ time—none of which address the refund policy question.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.50                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=554632;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.33s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_case_0: score=1.00, success=True
test_case_1: score=0.00, success=False


# Day 04 — Key Takeaways

Today we connected the complete basic evaluation flow:

```text
LLMTestCase
     ↓
Metric
     ↓
Judge
     ↓
Score + Reason
     ↓
Threshold
     ↓
PASS / FAIL